In [ ]:
import sys
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# we initialize the random generator with a seed, so we can reproduce results and debug
seed_value = 12345
np.random.seed(seed_value)

# set the radius of the disks, all have the same radius:
r = 2.0

# set the dimensions of the rectangle
a = 100.0
b = 100.0

# set the number of disks
n = 40

# set the number of dimensions
d = 2

# define an array of velocities
v = np.empty((n, d))

# define an array of positions (of the centers of the disks)
x = np.empty((n, d))

# print the values of the x and v vectors for each disk
def print_phase_space(x,v):
    for i in range(n):
        print(f"particle {i} is at ({x[i][0]:.2f},{x[i][1]:.2f}) with velocity ({v[i][0]:.2f},{v[i][1]:.2f}).")

# draw the disks inside the rectangle, with a velocity vector for each
def draw_all_disks(x,v):
    fig, ax = plt.subplots()

    # draw the disks
    for i in range(n):
        circle = plt.Circle((x[i][0], x[i][1]), r)
        ax.add_patch(circle)

    # draw the axes
    ax.set_xlim(0, a)
    ax.set_ylim(0, b)
    ax.set_aspect(1)

    # draw the velocities

    # prepare the quiver of arrows
    X = [x[i][0] for i in range(n)]
    Y = [x[i][1] for i in range(n)]
    U = [5*r*v[i][0] for i in range(n)]
    V = [5*r*v[i][1] for i in range(n)]
    
    plt.quiver(X, Y, U, V, edgecolor='k', facecolor='None', linewidth=.5)
    
    plt.show()

    # set the initial positions of the disks randomly
def is_far_enough(p,ps):
    for i in range(len(ps)):
        if ((p[0]-ps[i][0])**2+(p[1]-ps[i][1])**2)**(1/2) < 4:
            return False
        elif p[0] < r or p[0] > a-r or p[1] < r or p[1] > b-r:
            return False
    return True
def set_initial_positions(x):
    ps = []
    while len(ps) < x:
        p = np.random.rand( 2 )*np.array((a,b))
        if is_far_enough(p,ps):
            ps.append(p)
    return ps
print(set_initial_positions(40))

# - set the initial velocities of the disks randomly
def set_initial_velocities(vel):
    vs = []
    speed = 1.0
    while len(vs) < vel:
        angle = np.random.rand()*2*math.pi
        v = np.array((math.cos(angle), math.sin(angle)))*speed
        vs.append(v)
    return vs
print(set_initial_velocities(40))


# set the initial (step=0) positions, ensuring no overlaps
x = set_initial_positions(n)

# set the initial (step=0) velocities randomly
v = set_initial_velocities(n)

# draw the disks
draw_all_disks(x,v)

# find the first collision among the particles
# returns: time,i,j,wall_index
# wall_index is defined as follows.
# -1 if the first collision is between two disks, 
# 0 - left wall
# 1 - right wall
# 2 - bottom wall
# 3 - top wall.
# Also, j=-1 if the first collision is with a wall. In that case "i" is the particle that collided with the wall.
def is_any_collision(x):
    for i in range(len(x)):
        if x[i][0]-r < 0 :
            return (0,i,-1) 
        elif x[i][0]+r > a :
            return (1,i,0) 
        elif x[i][1]-r < 0 :
            return (2,i,-1)
        elif x[i][1]+r > b :
            return (3,i,-1)
        for j in range(i+1,len(x)):
            if ((x[i][0]-x[j][0])**2+(x[i][1]-x[j][1])**2)**(1/2) < 2*r:
                return (-1,i,j)
    return False
def time_to_earliest_collision(x,v):
    deltat = 0.1
    totalt = 0 
    p = x
    vt = v
    while True:
        a = is_any_collision(p)
        if str(a) != "False":
            for i in range(len(p)):
                p[i][0] = p[i][0] - vt[i][0]*totalt
                p[i][1] = p[i][1] - vt[i][1]*totalt
            return (totalt-deltat,a)
        for i in range(len(p)):
            p[i][0] = p[i][0] + vt[i][0]*deltat
            p[i][1] = p[i][1] + vt[i][1]*deltat
        totalt += deltat
    # -deltat because we want the time just before the collision
print(time_to_earliest_collision(x,v))

# Advance time t, assuming no collisions
def advance_time(t,x,v):
    new_x = x
    for i in range(len(x)):
        new_x[i][0] =  x[i][0] + np.array(v[i][0])*t
        new_x[i][1] =  x[i][1] + np.array(v[i][1])*t
    return new_x

def do_two_disk_collision(x, v, i, j):
    # the positions don't change
    x_new = [[x[p][k] for k in range(2)] for p in range(n)]
    v_new = [[v[p][k] for k in range(2)] for p in range(n)]

    norm = (np.array(x_new[j])-x_new[i])/np.linalg.norm(np.array(x_new[j])-x_new[i])
    v_new[i] = np.array(v_new[i]) - np.dot(v[i], norm) * norm+ np.dot(v[j], norm) * norm
    v_new[j] = np.array(v_new[j]) - np.dot(v[j], norm) * norm+ np.dot(v[i], norm) * norm
    return v_new

def do_wall_collision(x, v, i, wall_index):
    x_new = x
    v_new = v
    if wall_index == 0:
        v_new[i] = [-v[i][0],v[i][1]]
    elif wall_index == 1:
        v_new[i] = [-v[i][0],v[i][1]]
    elif wall_index == 2:
        v_new[i] = [v[i][0],-v[i][1]]
    elif wall_index == 3:
        v_new[i] = [v[i][0],-v[i][1]]    
    return v_new    

draw_all_disks(x,v)
# print_phase_space(x,v)

max_steps = 5000
for i_collision in range(max_steps):
    if i_collision % 500 == 0:
        draw_all_disks(x,v)
    infor = time_to_earliest_collision(x,v)
    new_list = advance_time(infor[0],x,v)
    x = new_list
    if infor[1][0] != -1:
        v = do_wall_collision(x, v, infor[1][1], infor[1][0])
        
    elif infor[1][0] == -1:
        v = do_two_disk_collision(x, v, infor[1][1], infor[1][2])


        
draw_all_disks(x,v)

#print_phase_space(x,v) 

def calc_v_stats(v):
    average_speed = 0
    v_squared = 0
    for i in range(len(v)):
        average_speed = average_speed + ( v[i][0]**2+ v[i][1]**2 )**(1/2)
        v_squared = v_squared + v[i][0]**2 + v[i][1]**2
    average_speed = average_speed/len(v)
    average_speed_squared = v_squared/len(v)
    variance_speed = average_speed_squared - average_speed**2
    return (average_speed, average_speed_squared, variance_speed)    
    
    
    return 
ave_speed, ave_speed_squared, var_speed = calc_v_stats(v)
print("Therefore, T is 1/2")
print(f"average speed <v> = {ave_speed}, the theoretical value is {((np.pi/4)**(1/2)):.3f}")
print(f"average speed squared <v^2> = {ave_speed_squared:.2f}, \n should be 1.0 because the collision of two disks just swap their velocities in the direction of their centers.\n Therefore, when we calculate the sum of all the squares of the speeds, it should remain constant.") 
print(f"variance of speed <v^2>-<v>^2 = {var_speed}, the theoretical value is {((4-math.pi)/4):.3f}")
print("There are some differences between the theoretical and simulated values because of the limited number of disks (n=40).")

from matplotlib.ticker import MultipleLocator
vnorm = np.sort(np.linalg.norm(v,axis = 1))[::-1]
vs =vnorm**2
cdf_1 = []
cumulative = 0
for i in range(len(vs)):
    cdf_1.append(1-cumulative-1/2/len(vs))
    cumulative+= ((1))/len(vs)
plt.scatter(vs,cdf_1, color = 'blue',label = "simulated CDF")
xax = np.linspace(0,max(vs),100)
cdf_2 = []
for k in xax:
    cdf_2.append(1-np.exp(-k))
plt.plot(xax,cdf_2,color = 'red',label = "theoretical CDF")
    # WRITE YOUR CODE FOR THE HISTOGRAM WITH SUPER-IMPOSED GRAPH HERE
plt.xlabel("v^2")
plt.ylabel("CDF")
plt.title("Cumulative Distribution Function of v^2")
plt.legend()


def evolve_system_with_stats(num_collisions, x, v):
    num_two_disk_collisions = 0
    current = 0 
    ave_v = 0
    ave_vs = 0 
    T = nested_list = [[0] for _ in range(40)]
    total_t_1 = 0 
    total_free_time = 0 
    total_free_path = 0
    sum_wall_momentum_transfer = [0,0,0,0]
    while current < num_collisions:
        infor = time_to_earliest_collision(x,v)
        new_list = advance_time(infor[0],x,v)
        x = new_list
        if infor[1][0] != -1:
            v = do_wall_collision(x, v, infor[1][1], infor[1][0])
            if infor[1][0] -2 < 0:
                sum_wall_momentum_transfer[infor[1][0]] = sum_wall_momentum_transfer[infor[1][0]] + 2*abs(v[infor[1][1]][0])
            else:
                sum_wall_momentum_transfer[infor[1][0]] = sum_wall_momentum_transfer[infor[1][0]] + 2*abs(v[infor[1][1]][1])
            for i in range(len(v)):
                ave_v += np.linalg.norm(v[i])*infor[0]
                ave_vs += np.linalg.norm(v[i])**2*infor[0]
            total_t_1 += infor[0]
        elif infor[1][0] == -1:
            v = do_two_disk_collision(x, v, infor[1][1], infor[1][2])
            total_t_1 += infor[0]
            T[infor[1][1]].append(total_t_1)
            T[infor[1][2]].append(total_t_1)
            total_free_time+=  (T[infor[1][1]][-1] - T[infor[1][1]][-2])**2
            total_free_time+=  (T[infor[1][2]][-1] - T[infor[1][2]][-2])**2
            total_free_path+= (T[infor[1][1]][-1] - T[infor[1][1]][-2])**2*np.linalg.norm(v[infor[1][1]])
            total_free_path+= (T[infor[1][2]][-1] - T[infor[1][2]][-2])**2*np.linalg.norm(v[infor[1][2]])
            for i in range(len(v)):
                ave_v += np.linalg.norm(v[i])*infor[0]
                ave_vs += np.linalg.norm(v[i])**2*infor[0]
            num_two_disk_collisions += 1
        current += 1
    for i in range(len(T)):
        T[i].append(total_t_1)
        total_free_time+=  (T[i][-1] - T[i][-2])**2
        total_free_path+= (T[i][-1] - T[i][-2])*np.linalg.norm(v[i])
    ave_mft = total_free_time/n/total_t_1/2
    ave_mfp = total_free_path/n/total_t_1/2
    ave_v = ave_v/n/total_t_1
    ave_vs = ave_vs/n/total_t_1
    print(f'the v-square is {ave_vs:03f}')
    return(total_t_1, sum_wall_momentum_transfer, ave_v, ave_mft, ave_mfp, num_two_disk_collisions)
(total_t_1, wall_momentum_transfer, ave_v, ave_mft, ave_mfp, num_two_disk_collisions) = evolve_system_with_stats(1000,x,v)
print(f"total time = {total_t_1}")
print(f"wall momentum transfer = {wall_momentum_transfer}")
print(f"average speed = {ave_v}")
print(f"average mean free time = {ave_mft}")
print(f"average mean free path = {ave_mfp}")
print(f"number of two disk collisions = {num_two_disk_collisions}")
